# Pipeline Tag Prediction

## Model A: RoBERTa-base
**DATASCI 266: Natural Language Processing with Deep Learning**

UC Berkeley, School of Information

---

This notebook implements an improvement model over the baseline using roBERTa-base with sliding windows. It runs in this order:

1. Load split dataframes used in baseline
2. Setup tokenizer and sliding window
3. Setup HuggingFace Datasets
5. RoBERTa model
6. Setup training configuration and train model
7. Predictions and metric analysis

## 0. Setup

In [ ]:
!pip install -q -U evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
import sys

In [ ]:
!{sys.executable} -m pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.0 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
import numpy as np
import evaluate
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
import warnings, re, ast
from datasets import load_dataset, DatasetDict, Dataset


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from transformers import RobertaTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer

warnings.filterwarnings('ignore')

# Plot defaults
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (12, 5)})

print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Load DataFrames

Import the train, test, and validation dataframes that were used for the baseline. They underwent an 80/20/20 split. This ensures we are using the same exact dataset across all models to best inform our model and error analysis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, json

DRIVE_DIR_2 = '/content/drive/MyDrive/266-pipeline-tag-prediction'

train_df = pd.read_parquet(f'{DRIVE_DIR_2}/train_df.parquet')
val_df = pd.read_parquet(f'{DRIVE_DIR_2}/val_df.parquet')
test_df = pd.read_parquet(f'{DRIVE_DIR_2}/test_df.parquet')

metadata = json.load(open(f'{DRIVE_DIR_2}/metadata.json'))
label2id, id2label = metadata['label2id'], metadata['id2label']

Mounted at /content/drive


In [ ]:
print("Shape of train:", train_df.shape)
print("Shape of validation:", val_df.shape)
print("Shape of test:", test_df.shape)

Shape of train: (112828, 7)
Shape of validation: (14104, 7)
Shape of test: (14104, 7)


## 2. Sliding Window & Tokenizer

Setup the sliding window to increase text input coverage in the roBERTa model. The context window for the model has a maximum length of 512 tokens. From the data exploration, most of the model cards per model have tokens that would fit into the context window, but there is still a generous amount of cards that would benefit from the expansion. We choose to set 2 sliding windows per model card document to increase text coverage compared to the base iteration where we only use the given maximum of 512 tokens without a sliding window; but this still misses outliers. We began with 10 sliding windows to cover the entire text, but this caused runtime crashes on the given GPU setup and we decided to experiment with smaller window counts to see if there were any realized benefits to increasing them.

In [ ]:
# Create `doc_id` to map iterations of each sliding window back to the original
# model card document/text

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df['doc_id'] = train_df.index
val_df['doc_id'] = val_df.index
test_df['doc_id'] = test_df.index

In [ ]:
# Analyze lengths of model card text

lengths = train_df["text"].str.split().str.len()
print(lengths.describe())
print(lengths.sort_values(ascending=False).head(20))

count    112828.000000
mean        502.763924
std         872.165612
min           1.000000
25%         115.000000
50%         234.000000
75%         536.000000
max       15633.000000
Name: text, dtype: float64
74996     15633
4745      15058
9345      14660
9985      14492
103140    14414
55884     13924
62341     13558
17102     13543
38819     13535
48835     13533
78517     13526
86420     13521
107302    13511
77213     13481
59867     13481
110231    13073
55380     12881
75301     12785
99094     12729
23820     12693
Name: text, dtype: int64


In [ ]:
for t in [1000, 2000, 3000, 5000, 8000]:
    print(f">{t} words:", (lengths > t).sum())

>1000 words: 13369
>2000 words: 4687
>3000 words: 2381
>5000 words: 866
>8000 words: 229


In [ ]:
# Load tokenizer from roBERTa-base

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
# Set max sliding windows per model card doc
max_windows_per_doc_id = 2

def tokenize_sliding_window(batch):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512, # max context/sliding window length
        stride=128, # allow 128 tokens to overlap between sliding windows for better performance
        return_overflowing_tokens=True
    )
    window2data_map = tokenized.pop("overflow_to_sample_mapping")
    comp = {}
    id_1 = []
    # if-else checks for sliding window needs per doc
    for i, doc_id_1 in enumerate(window2data_map):
      x = comp.get(doc_id_1, 0)
      if x < max_windows_per_doc_id:
        id_1.append(i)
        comp[doc_id_1] = x + 1

    tokenized_2 = {k: [v[i] for i in id_1] for k, v in tokenized.items()}
    tokenized_2["label"] = [batch["label"][window2data_map[i]] for i in id_1]
    tokenized_2["doc_id"] = [batch["doc_id"][window2data_map[i]] for i in id_1]

    return tokenized_2

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## 3. Convert DataFrames into Hugging Face Dataset objects

Setup the data to be usable by the tokenizer, model, and trainer.

In [ ]:
# Convert DataFrames to Hugging Face Datasets

train_dataset = Dataset.from_pandas(train_df[["text", "label", "doc_id"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label", "doc_id"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label", "doc_id"]])

In [ ]:
# Apply sliding window splits to each dataset

train_dataset = train_dataset.map(tokenize_sliding_window,
                                  batched=True,
                                  batch_size=32,
                                  remove_columns=train_dataset.column_names, # drop untokenized cols
                                  load_from_cache_file=False, # fresh dataset per run
                                  writer_batch_size=200 # small fix for RAM usage limits
                                  )
val_dataset = val_dataset.map(tokenize_sliding_window,
                              batched=True,
                              batch_size=32,
                              remove_columns=val_dataset.column_names,
                              load_from_cache_file=False,
                              writer_batch_size=200
                              )
test_dataset = test_dataset.map(tokenize_sliding_window,
                                batched=True,
                                batch_size=32,
                                remove_columns=test_dataset.column_names,
                                load_from_cache_file=False,
                                writer_batch_size=200
                                )

Map:   0%|          | 0/112828 [00:00<?, ? examples/s]

Map:   0%|          | 0/14104 [00:00<?, ? examples/s]

Map:   0%|          | 0/14104 [00:00<?, ? examples/s]

In [ ]:
# Check how many docs to windows were generated per dataset

print(f"train: {len(train_df)} docs -> {len(train_dataset)} windows")
print(f"val:   {len(val_df)} docs -> {len(val_dataset)} windows")
print(f"test:  {len(test_df)} docs -> {len(test_dataset)} windows")

train: 112828 docs -> 185457 windows
val:   14104 docs -> 23172 windows
test:  14104 docs -> 23130 windows


## 4. RoBERTa Model

Prepare datasets for the model and load the roBERTa model.

In [ ]:
# Setup data as PyTorch tensors for model inputs and preserves all columns

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
    output_all_columns=True
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
    output_all_columns=True
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"],
    output_all_columns=True
)

In [ ]:
# Load model

num_labels = len(label2id)
roBERTa_model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=num_labels)

# Setup metrics - accuracy and F1
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Train

Setup training argument parameters and the trainer.

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True
)

trainer = Trainer(
    model=roBERTa_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

Uncomment the block below to check GPU usage while training is running.

In [ ]:
# import subprocess, threading, time

# def log_gpu(interval=5, duration=60):
#     end = time.time() + duration
#     while time.time() < end:
#         out = subprocess.run(
#             ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
#             capture_output=True, text=True
#         )
#         print(out.stdout.strip())
#         time.sleep(interval)

# threading.Thread(target=log_gpu, args=(5, 60)).start()

In [ ]:
# Run train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.181980,0.172504,0.947566,0.942465


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5796, training_loss=0.28607119684634, metrics={'train_runtime': 4644.1801, 'train_samples_per_second': 39.933, 'train_steps_per_second': 1.248, 'total_flos': 4.879929193535693e+16, 'train_loss': 0.28607119684634, 'epoch': 1.0})

In [ ]:
# Evaluate training
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.181980,0.172504,1,0.947566,0.942465


{'eval_loss': 0.17250372469425201,
 'eval_accuracy': 0.9475660279647851,
 'eval_macro_f1': 0.9424648559061122}

## 6. Predictions and Metric Analysis

In [ ]:
test_predictions = trainer.predict(test_dataset)

In [ ]:
window_logits = test_predictions.predictions  # shape: (num_windows, num_labels)
window_doc_ids = np.array(test_dataset["doc_id"])

In [24]:
# Aggregate metrics across the same doc_id for the multiple sliding windows
# used max pooling and not mean pooling to avoid dilution across windows

unique_doc_ids = np.unique(window_doc_ids)
# avg_logits = np.zeros((len(unique_doc_ids), window_logits.shape[1]))
max_logits = np.zeros((len(unique_doc_ids), window_logits.shape[1]))

for row, doc_id in enumerate(unique_doc_ids):
    mask = window_doc_ids == doc_id
    # avg_logits[row] = window_logits[mask].mean(axis=0)
    max_logits[row] = window_logits[mask].max(axis=0)

# Take max among preds across windows for the pred per doc
# preds = avg_logits.argmax(axis=1)
preds = max_logits.argmax(axis=1)

# Actual document-level labels per unique_doc_ids
doc_true_labels = (test_df.set_index("doc_id").loc[unique_doc_ids, "label"].values)

# Print check for final aggregated prediction counts
print(f"{len(test_df)} documents -> {len(test_dataset)} windows "
      f"-> {len(unique_doc_ids)} aggregated predictions")

14104 documents -> 23130 windows -> 14104 aggregated predictions


In [25]:
# Classification Report

target_names = [id2label[str(i)] for i in range(len(id2label))]

print(classification_report(
    doc_true_labels,
    preds,
    target_names=target_names
))

                              precision    recall  f1-score   support

automatic-speech-recognition       0.98      0.98      0.98       617
        image-classification       0.97      0.95      0.96       331
          image-text-to-text       0.88      0.85      0.86      1240
                    robotics       0.96      0.97      0.96       347
         sentence-similarity       0.97      0.98      0.97       343
         text-classification       0.95      0.92      0.94      1033
             text-generation       0.96      0.97      0.96      6689
               text-to-image       1.00      0.99      0.99      2239
        token-classification       0.96      0.93      0.94       362
                 translation       0.97      0.96      0.97       903

                    accuracy                           0.96     14104
                   macro avg       0.96      0.95      0.95     14104
                weighted avg       0.96      0.96      0.96     14104

